# NB04 — MOFA+ Multi-Omic Integration, FDA & Abundance Trajectory
**Pipeline:** Generalised Metabolite–Metagenomics Correlation Study  
**Inputs:** `preprocessed_data.pkl` + `analysis_results.pkl`  
**Outputs:** `mofa_results.pkl`, factor plots, trajectory figures

| Step | Analysis |
|---|---|
| 1 | Setup and data load |
| 2 | MOFA+ joint factorisation (TruncatedSVD fallback for negative CLR) |
| 3 | Factor–stage associations (Kruskal-Wallis + BH) |
| 4 | Variance explained per factor (R² with ordinal stage) |
| 5 | Top feature loadings per CRC-associated factor |
| 6 | PERMANOVA — beta diversity significance (Aitchison distance) |
| 7 | FDA (LDA) — 3-group stage separation, 10-fold CV |
| 8 | Metabolite abundance trajectory across 6 CRC stages |
| 9 | MP (Metachronous Polyp) recovery trajectory |
| 10 | Save MOFA results |


## 1 · Setup

In [1]:
import sys, warnings, inspect
from pathlib import Path
warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path(".").resolve()))

from utils import (
    INTER_DIR, FIG_DIR, TABLE_DIR,
    DATASET_PRIMARY, CRC_STAGE_ORDER, THREE_GROUP_ORDER,
    FDR_THRESHOLD, annotate_pathway,
    load_pickle, save_pickle, savefig,
    PALETTE_STAGE6, PALETTE_3GROUP,
)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy.stats import spearmanr, kruskal
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from statsmodels.stats.multitest import multipletests

pre = load_pickle(INTER_DIR / "preprocessed_data.pkl")
ana = load_pickle(INTER_DIR / "analysis_results.pkl")

sp_clr = ana["spe"]
mt_log = ana["mtb"]
meta_y = ana["meta_yk"]
nm_y   = ana["nm_y"]

# Sanitise name map: replace NaN/float values with the KEGG ID string
nm_y = {k: (v if isinstance(v, str) else k) for k, v in nm_y.items()}

for d in [FIG_DIR/"mofa", FIG_DIR/"trajectory", TABLE_DIR]:
    d.mkdir(parents=True, exist_ok=True)
print("NB04 setup complete.")


Loaded: C:\Users\andna\Documents\KI\Results\intermediate\preprocessed_data.pkl
Loaded: C:\Users\andna\Documents\KI\Results\intermediate\analysis_results.pkl
NB04 setup complete.


---
## 2 · MOFA+ Joint Factor Analysis

In [2]:
# MOFA+ version guard: API changed significantly between 0.6.x and 0.7.x
MOFA_AVAILABLE = False
MOFA_VERSION   = None

try:
    import mofapy2
    from mofapy2.run.entry_point import entry_point
    MOFA_VERSION = getattr(mofapy2, "__version__", "unknown")
    sig = inspect.signature(entry_point().set_data_matrix)
    MOFA_AVAILABLE = True
    print(f"mofapy2 v{MOFA_VERSION} available — using MOFA+ factorisation.")
except ImportError:
    print("mofapy2 not installed. Falling back to TruncatedSVD.")
    print("  Install: pip install mofapy2==0.7.0")
except Exception as e:
    print(f"mofapy2 v{MOFA_VERSION} API check failed: {e}  ->  falling back to TruncatedSVD")



        #########################################################
        ###           __  __  ____  ______                    ### 
        ###          |  \/  |/ __ \|  ____/\    _             ### 
        ###          | \  / | |  | | |__ /  \ _| |_           ### 
        ###          | |\/| | |  | |  __/ /\ \_   _|          ###
        ###          | |  | | |__| | | / ____ \|_|            ###
        ###          |_|  |_|\____/|_|/_/    \_\              ###
        ###                                                   ### 
        ######################################################### 
         


mofapy2 v0.7.3 available — using MOFA+ factorisation.


In [3]:
n_factors = 15
samples   = list(sp_clr.index)
method_used = None  # will be set to "MOFA+" or "TruncatedSVD fallback"

if MOFA_AVAILABLE:
    sp_mat = sp_clr.values.astype(float)       # (samples, features)
    mt_mat = mt_log.values.astype(float)       # (samples, features)

    ent = entry_point()
    ent.set_data_options(scale_groups=False, scale_views=True)
    ent.set_data_matrix(
        [[sp_mat], [mt_mat]],       # outer=views, inner=groups
        likelihoods=["gaussian", "gaussian"],
        views_names=["species", "metabolites"],
        samples_names=[samples],
        features_names=[list(sp_clr.columns), list(mt_log.columns)]
    )
    ent.set_model_options(factors=n_factors, spikeslab_weights=True)
    ent.set_train_options(iter=1000, convergence_mode="fast",
                           startELBO=1, freqELBO=5, seed=42, verbose=False)
    ent.build(); ent.run()

    Z    = ent.model.nodes["Z"].getExpectations()["E"]
    W_sp = ent.model.nodes["W"].getExpectations()[0]["E"]
    W_mt = ent.model.nodes["W"].getExpectations()[1]["E"]
    mofa_Z             = pd.DataFrame(Z,    index=samples,
                                       columns=[f"Factor{j+1}" for j in range(Z.shape[1])])
    mofa_W_species     = pd.DataFrame(W_sp, index=sp_clr.columns, columns=mofa_Z.columns)
    mofa_W_metabolites = pd.DataFrame(W_mt, index=mt_log.columns, columns=mofa_Z.columns)
    method_used = "MOFA+"
    print(f"MOFA+ complete: {Z.shape[1]} factors, {len(samples)} samples")

else:
    # TruncatedSVD fallback — handles negative CLR values (NMF cannot)
    from sklearn.decomposition import TruncatedSVD
    from sklearn.preprocessing import StandardScaler

    sp_sc = StandardScaler().fit_transform(sp_clr.values)
    mt_sc = StandardScaler().fit_transform(mt_log.values)
    X_cat = np.hstack([sp_sc, mt_sc])

    svd = TruncatedSVD(n_components=n_factors, random_state=42)
    Z   = svd.fit_transform(X_cat)
    mofa_Z = pd.DataFrame(
        Z, index=sp_clr.index,
        columns=[f"Factor{j+1}" for j in range(n_factors)])

    n_sp = sp_clr.shape[1]
    mofa_W_species     = pd.DataFrame(
        svd.components_[:, :n_sp].T,
        index=sp_clr.columns, columns=mofa_Z.columns)
    mofa_W_metabolites = pd.DataFrame(
        svd.components_[:, n_sp:].T,
        index=mt_log.columns, columns=mofa_Z.columns)
    method_used = "TruncatedSVD fallback"
    print(f"TruncatedSVD fallback: {n_factors} factors, "
          f"variance explained = {svd.explained_variance_ratio_.sum():.3f}")



        #########################################################
        ###           __  __  ____  ______                    ### 
        ###          |  \/  |/ __ \|  ____/\    _             ### 
        ###          | \  / | |  | | |__ /  \ _| |_           ### 
        ###          | |\/| | |  | |  __/ /\ \_   _|          ###
        ###          | |  | | |__| | | / ____ \|_|            ###
        ###          |_|  |_|\____/|_|/_/    \_\              ###
        ###                                                   ### 
        ######################################################### 
         


Scaling views to unit variance...

Groups names not provided, using default naming convention:
- group1, group2, ..., groupG

Successfully loaded view='species' group='group0' with N=347 samples and D=500 features...
Successfully loaded view='metabolites' group='group0' with N=347 samples and D=150 features...


Model options:
- Automatic Relevance Determination prior on the factors: F

---
## 3 · Factor–Stage Associations (Kruskal-Wallis + BH)

In [4]:
meta_aligned  = meta_y.loc[mofa_Z.index]
stage3        = meta_aligned["Stage.3Group"].dropna()
common_smp    = stage3.index.intersection(mofa_Z.index)
Z_aligned     = mofa_Z.loc[common_smp]
stage_aligned = stage3.loc[common_smp]

factor_pvals = {}
for fac in mofa_Z.columns:
    groups = [
        Z_aligned.loc[stage_aligned == g, fac].dropna()
        for g in THREE_GROUP_ORDER
        if g in stage_aligned.values
    ]
    groups = [g for g in groups if len(g) > 2]
    if len(groups) >= 2:
        stat, pval = kruskal(*groups)
        factor_pvals[fac] = pval

_, qvals, _, _ = multipletests(list(factor_pvals.values()), method="fdr_bh")
factor_assoc = pd.DataFrame({
    "factor": list(factor_pvals.keys()),
    "pval":   list(factor_pvals.values()),
    "qval":   qvals
}).sort_values("pval")
sig_factors = factor_assoc[factor_assoc["qval"] <= FDR_THRESHOLD]["factor"].tolist()

print(f"Stage-associated factors (q<0.05): {sig_factors}")
factor_assoc.to_csv(TABLE_DIR / "mofa_factor_stage_association.csv", index=False)
print(factor_assoc.head(8).to_string(index=False))

# Boxplots for top CRC-associated factors
plot_factors = sig_factors[:4] if sig_factors else mofa_Z.columns[:4].tolist()
fig, axes = plt.subplots(1, len(plot_factors), figsize=(4*len(plot_factors), 4),
                          sharey=False)
if len(plot_factors) == 1:
    axes = [axes]
for ax, fac in zip(axes, plot_factors):
    bdata  = [Z_aligned.loc[stage_aligned == g, fac].dropna()
               for g in THREE_GROUP_ORDER if g in stage_aligned.values]
    blabels = [g for g in THREE_GROUP_ORDER if g in stage_aligned.values]
    bp = ax.boxplot(bdata, labels=blabels, patch_artist=True, notch=False)
    for patch, g in zip(bp["boxes"], blabels):
        patch.set_facecolor(PALETTE_3GROUP.get(g, "#aaa"))
    ax.set_title(fac, fontsize=10)
    ax.set_xticklabels(blabels, rotation=30, ha="right")
    ax.set_ylabel("Factor value")
plt.suptitle("MOFA+ Factors Associated with CRC Stage", fontweight="bold")
plt.tight_layout()
savefig(fig, "mofa", "nb04_mofa_factor_boxplots.png")


Stage-associated factors (q<0.05): ['Factor7', 'Factor4', 'Factor14']
  factor     pval     qval
 Factor7 0.000849 0.007334
 Factor4 0.000978 0.007334
Factor14 0.006772 0.033860
Factor12 0.017041 0.063903
 Factor3 0.022531 0.067592
Factor10 0.027916 0.069789
Factor15 0.057129 0.122419
 Factor5 0.218318 0.394186
Saved figure: C:\Users\andna\Documents\KI\Results\figures\mofa\nb04_mofa_factor_boxplots.png


---
## 4 · Variance Explained per Factor (R² with ordinal CRC stage)

In [5]:
# NB04-5 (UPGRADED): Ordinal regression — OrderedModel + McFadden pseudo-R2
# Replaces: Spearman rho^2 on Stage.Num (treated stages as continuous)
# Correct treatment: ordinal logit via statsmodels OrderedModel
import warnings
from statsmodels.miscmodels.ordinal_model import OrderedModel

# Build ordered Categorical from Stage.6 values
# FIX 1: wrap pd.Categorical in pd.Series with the original index.
# OrderedModel requires a pandas Series; a bare pd.Categorical causes numpy
# to cast it to dtype=object, which statsmodels rejects with:
#   "ValueError: Pandas data cast to numpy dtype of object."
valid_ord = meta_y['Stage.6'].notna() & meta_y.index.isin(mofa_Z.index)
y_ord_raw = meta_y.loc[valid_ord, 'Stage.6']
y_ord = pd.Series(
    pd.Categorical(y_ord_raw, categories=CRC_STAGE_ORDER, ordered=True),
    index=y_ord_raw.index,
    name='Stage.6',
)
Z_ve = mofa_Z.loc[valid_ord]
print(f'Samples for ordinal regression: {valid_ord.sum()}')
print('Stage distribution:')
print(y_ord_raw.value_counts().reindex(CRC_STAGE_ORDER).fillna(0).astype(int))

# Null model log-likelihood for McFadden R² denominator.
# OrderedModel forbids a constant column (thresholds serve as intercepts).
# Analytical solution: for ordered logit the null MLE sets each predicted P(Y=j)
# equal to the empirical proportion p_j, so LL_null = Σ_j n_j·log(p_j).
from collections import Counter as _Counter
_cnts = _Counter(y_ord)
_N    = len(y_ord)
ll_null = float(np.sum(
    [_cnts[s] * np.log(_cnts[s] / _N)
     for s in CRC_STAGE_ORDER if _cnts.get(s, 0) > 0]
))
print(f'Null model log-likelihood (analytical): {ll_null:.3f}')

# Per-factor ordinal regression
# FIX 3: slice Z_ve[[fac]] (keeps sample index) instead of Z_ve[fac].values
# (.values strips the index, causing an index mismatch with y_ord).
ordinal_rows = []
for fac in mofa_Z.columns:
    x_fac = Z_ve[[fac]]   # DataFrame column slice — preserves sample index
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        try:
            res = OrderedModel(y_ord, x_fac, distr='logit').fit(method='bfgs', disp=False)
            mcfadden_r2 = max(0.0, 1.0 - res.llf / ll_null)
            pval = float(res.pvalues.get(fac, 1.0))
        except Exception as _e:
            mcfadden_r2, pval = float('nan'), 1.0
    ordinal_rows.append({'factor': fac, 'mcfadden_r2': mcfadden_r2, 'pval_factor': pval})

ordinal_df = pd.DataFrame(ordinal_rows).sort_values('mcfadden_r2', ascending=False)
_, ord_qvals, _, _ = multipletests(ordinal_df['pval_factor'].fillna(1), method='fdr_bh')
ordinal_df['qval_factor'] = ord_qvals
ordinal_df.to_csv(TABLE_DIR / 'mofa_ordinal_regression_results.csv', index=False)

# Backward-compat alias so downstream cells that use var_exp_df still work
var_exp_df = ordinal_df.rename(columns={
    'mcfadden_r2': 'rho2_stage',
    'pval_factor': 'pval_stage',
    'qval_factor': 'qval_stage',
})
var_exp_df['rho_stage'] = float('nan')  # not applicable for ordinal model

# Bar chart — top 12 factors by McFadden pseudo-R2
top_ve = ordinal_df.head(12)
colors = ['#F44336' if q < FDR_THRESHOLD else '#90CAF9' for q in top_ve['qval_factor']]

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(top_ve['factor'], top_ve['mcfadden_r2'], color=colors, edgecolor='white', linewidth=0.5)
ax.set_xlabel('MOFA+ Factor')
ax.set_ylabel('McFadden pseudo-R² (ordinal logit)')
ax.set_title(
    f'Factor Association with CRC Stage Progression — Ordinal Regression ({method_used})',
    fontweight='bold')
ax.set_xticklabels(top_ve['factor'], rotation=35, ha='right')
ax.legend(handles=[
    mpatches.Patch(color='#F44336', label='Stage-significant (q<0.05)'),
    mpatches.Patch(color='#90CAF9', label='Non-significant'),
], fontsize=8)
plt.tight_layout()
savefig(fig, 'mofa', 'nb04_mofa_ordinal_variance_explained.png')
print(ordinal_df.head(8)[['factor','mcfadden_r2','pval_factor','qval_factor']].to_string(index=False))
print('Saved: mofa_ordinal_regression_results.csv')


Samples for ordinal regression: 347
Stage distribution:
Stage.6
Healthy         127
Stage_0          27
HS               30
Stage_I_II       69
Stage_III_IV     54
MP               40
Name: count, dtype: int64
Null model log-likelihood (analytical): -568.367
Saved figure: C:\Users\andna\Documents\KI\Results\figures\mofa\nb04_mofa_ordinal_variance_explained.png
  factor  mcfadden_r2  pval_factor  qval_factor
Factor14     0.010359     0.000670     0.010044
 Factor4     0.006384     0.007214     0.044616
 Factor7     0.006067     0.008923     0.044616
Factor15     0.005339     0.014007     0.052528
 Factor3     0.002149     0.118349     0.355046
 Factor8     0.001751     0.159970     0.386342
Factor10     0.001583     0.180293     0.386342
Factor13     0.001169     0.248467     0.465875
Saved: mofa_ordinal_regression_results.csv


---
## 5 · Top Feature Loadings per CRC-Associated Factor

In [6]:
loading_rows = []
plot_sig_facs = sig_factors[:6] if sig_factors else mofa_Z.columns[:6].tolist()

for fac in plot_sig_facs:
    for feat, w in mofa_W_species[fac].abs().nlargest(10).items():
        loading_rows.append({"factor": fac, "feature": feat,
                              "weight": mofa_W_species.loc[feat, fac],
                              "abs_weight": abs(mofa_W_species.loc[feat, fac]),
                              "view": "species"})
    for feat, w in mofa_W_metabolites[fac].abs().nlargest(10).items():
        loading_rows.append({"factor": fac, "feature": feat,
                              "name": nm_y.get(feat, feat),
                              "weight": mofa_W_metabolites.loc[feat, fac],
                              "abs_weight": abs(mofa_W_metabolites.loc[feat, fac]),
                              "view": "metabolite",
                              "pathway": annotate_pathway(feat)})

loadings_df = pd.DataFrame(loading_rows)
loadings_df.to_csv(TABLE_DIR / "mofa_top_loadings.csv", index=False)

# Heatmap of metabolite loadings
fac_cols = plot_sig_facs
if fac_cols and not mofa_W_metabolites.empty:
    mt_load  = mofa_W_metabolites[fac_cols]
    top_feats = mt_load.abs().max(axis=1).nlargest(30).index
    mt_load_top = mt_load.loc[top_feats].copy()
    mt_load_top.index = [nm_y.get(f, f)[:30] for f in top_feats]
    fig, ax = plt.subplots(figsize=(8, 9))
    sns.heatmap(mt_load_top, cmap="RdBu_r", center=0, linewidths=0.3, ax=ax,
                cbar_kws={"label": "MOFA+ weight"})
    ax.set_title("Top Metabolite Loadings — CRC-Associated MOFA+ Factors", fontweight="bold")
    plt.tight_layout()
    savefig(fig, "mofa", "nb04_mofa_loading_heatmap.png")


Saved figure: C:\Users\andna\Documents\KI\Results\figures\mofa\nb04_mofa_loading_heatmap.png


---
## 6 · PERMANOVA — Beta Diversity Significance
Tests whether CRC stage significantly explains microbial community structure (Aitchison distance).

> **Note on analysis order:** Scientifically, PERMANOVA should precede MOFA factorisation
> to first confirm that CRC stage explains significant beta-diversity variance — that
> observation justifies fitting a joint factor model. The ordering here reflects notebook
> execution flow; for manuscript Methods, present PERMANOVA before MOFA.
> NaN-stage samples are excluded from both the distance matrix and group labels so they
> do not inflate group count or bias pseudo-F.

In [7]:
perm_pval = None
perm_stat = None

try:
    from skbio.stats.distance import DistanceMatrix, permanova
    from scipy.spatial.distance import pdist, squareform

    # NB04-3: exclude NaN-stage samples from both distance matrix and group labels.
    # Including them as "Unknown" inflates group count and biases pseudo-F.
    stage3g_full = meta_y.loc[sp_clr.index, "Stage.3Group"].dropna()
    valid_perm   = stage3g_full.index                          # samples with known stage
    sp_perm      = sp_clr.loc[valid_perm]                      # subset species matrix
    grp_labels   = stage3g_full.values                         # no NaN / "Unknown"

    # Aitchison distance = CLR Euclidean (sp_perm is already CLR-transformed)
    dm_values    = squareform(pdist(sp_perm.values, metric="euclidean"))
    dm           = DistanceMatrix(dm_values, ids=list(valid_perm))

    result_perm  = permanova(dm, grp_labels, permutations=999)
    perm_stat    = result_perm["test statistic"]
    perm_pval    = result_perm["p-value"]
    print(f"PERMANOVA (Stage.3Group ~ Aitchison distance, n={len(valid_perm)}):")
    print(f"  Pseudo-F = {perm_stat:.3f},  p = {perm_pval:.4f} (999 permutations)")
    pd.DataFrame([{"pseudo_F": perm_stat, "p_value": perm_pval, "permutations": 999,
                   "n_samples": len(valid_perm),
                   "distance": "Aitchison", "grouping": "Stage.3Group"}]
                 ).to_csv(TABLE_DIR / "permanova_results.csv", index=False)

except ImportError:
    print("scikit-bio not installed — pip install scikit-bio")
    print("Skipping PERMANOVA.")
except Exception as e:
    print(f"PERMANOVA error: {e}")


PERMANOVA (Stage.3Group ~ Aitchison distance, n=347):
  Pseudo-F = 1.235,  p = 0.0580 (999 permutations)


---
## 7 · FDA — Fisher Discriminant Analysis for Stage Separation

In [8]:
from sklearn.preprocessing import StandardScaler as _FDAScaler

# NB04-2: scale each modality separately before concatenation.
# Without this, LDA's within-class covariance is dominated by whichever modality
# has higher raw variance, producing biased discriminant axes.
fda_common = common_smp.intersection(sp_clr.index).intersection(mt_log.index)
X_sp_raw   = sp_clr.loc[fda_common].values
X_mt_raw   = mt_log.loc[fda_common].values

y_stage    = stage_aligned.loc[fda_common].map(
    {"Healthy": 0, "Early_CRC": 1, "Advanced_CRC": 2})

cv_fda     = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
fda_scores = []

# Per-fold scaling to avoid test-fold leakage
for train_idx, test_idx in cv_fda.split(X_sp_raw, y_stage.values):
    sp_scaler = _FDAScaler()
    mt_scaler = _FDAScaler()
    sp_tr = sp_scaler.fit_transform(X_sp_raw[train_idx])
    sp_te = sp_scaler.transform(X_sp_raw[test_idx])
    mt_tr = mt_scaler.fit_transform(X_mt_raw[train_idx])
    mt_te = mt_scaler.transform(X_mt_raw[test_idx])
    X_tr  = np.hstack([sp_tr, mt_tr])
    X_te  = np.hstack([sp_te, mt_te])
    lda_cv = LinearDiscriminantAnalysis(n_components=2, solver="svd")
    lda_cv.fit(X_tr, y_stage.values[train_idx])
    preds = lda_cv.predict(X_te)
    fda_scores.append(balanced_accuracy_score(y_stage.values[test_idx], preds))

fda_bal_acc = float(np.mean(fda_scores))
print(f"FDA 10-fold CV balanced accuracy: {fda_bal_acc:.3f} +/- {np.std(fda_scores):.3f}")

# Full-data fit for visualisation only (not used for reported accuracy)
sp_sc_full = _FDAScaler().fit_transform(X_sp_raw)
mt_sc_full = _FDAScaler().fit_transform(X_mt_raw)
X_combined = np.hstack([sp_sc_full, mt_sc_full])
lda_full = LinearDiscriminantAnalysis(n_components=2, solver="svd")
lda_full.fit(X_combined, y_stage.values)
fda_coords = lda_full.transform(X_combined)

fig, ax = plt.subplots(figsize=(7, 5))
for g_label, g_num in [("Healthy", 0), ("Early_CRC", 1), ("Advanced_CRC", 2)]:
    mask = y_stage.values == g_num
    ax.scatter(
        fda_coords[mask, 0],
        fda_coords[mask, 1] if fda_coords.shape[1] > 1 else np.zeros(mask.sum()),
        c=PALETTE_3GROUP.get(g_label, "#999"), label=g_label, s=18, alpha=0.7
    )
ax.set_title(f"FDA — CRC Stage Separation  (Bal. Acc. = {fda_bal_acc:.2f})", fontweight="bold")
ax.set_xlabel("LD1"); ax.set_ylabel("LD2"); ax.legend()
plt.tight_layout()
savefig(fig, "mofa", "nb04_fda_stage_separation.png")


FDA 10-fold CV balanced accuracy: 0.361 +/- 0.065
Saved figure: C:\Users\andna\Documents\KI\Results\figures\mofa\nb04_fda_stage_separation.png


---
## 8 · Metabolite Abundance Trajectory Across 6 CRC Stages

In [9]:
traj_df = pd.DataFrame()

da_adv     = ana["da_mtb"].get("Healthy_vs_Advanced_CRC", pd.DataFrame())
sig_metabs = (
    da_adv[da_adv["qval"] <= FDR_THRESHOLD]["feature"].tolist()
    if not da_adv.empty else []
)
# Also include early-stage dysregulated
da_early   = ana["da_mtb"].get("Healthy_vs_Early_CRC", pd.DataFrame())
sig_early  = (
    da_early[da_early["qval"] <= FDR_THRESHOLD]["feature"].tolist()
    if not da_early.empty else []
)
sig_metabs = list(dict.fromkeys(sig_metabs + sig_early))
sig_metabs = [m for m in sig_metabs if m in mt_log.columns][:25]

if sig_metabs:
    # Use full YACHIDA metadata (not feature-selected) for 6-stage breakdown
    meta_full  = pre["harmonized_meta"].get(DATASET_PRIMARY, pd.DataFrame())
    mt_full    = pre["mtb_log10"].get(DATASET_PRIMARY, pd.DataFrame())
    stage6     = meta_full["Stage.6"].dropna() if not meta_full.empty else pd.Series(dtype=str)
    common_6   = stage6.index.intersection(mt_full.index)
    mt_stage   = mt_full.loc[common_6]
    s6_aligned = stage6.loc[common_6]

    traj = {}
    for m in sig_metabs:
        if m not in mt_stage.columns:
            continue
        row = {}
        for s in CRC_STAGE_ORDER:
            vals = mt_stage.loc[s6_aligned == s, m].dropna()
            row[s] = vals.median() if len(vals) else np.nan
        traj[m] = row                              # KEGG ID as key (unique)
    traj_df = pd.DataFrame(traj).T
    traj_df.index.name = "kegg_id"

    # Plot top 10 by range
    ranges   = traj_df.max(axis=1) - traj_df.min(axis=1)
    top_traj = traj_df.loc[ranges.nlargest(10).index]

    cmap = plt.cm.get_cmap("tab10", len(top_traj))
    fig, ax = plt.subplots(figsize=(10, 5))
    for i, (kegg_id, row) in enumerate(top_traj.iterrows()):
        vals = row.reindex(CRC_STAGE_ORDER)
        ax.plot(CRC_STAGE_ORDER, vals, marker="o",
                label=nm_y.get(kegg_id, kegg_id), color=cmap(i), lw=2)
    ax.set_xlabel("CRC Stage")
    ax.set_ylabel("Median abundance (log10)")
    ax.set_title("Metabolite Abundance Trajectory Across CRC Stages", fontweight="bold")
    ax.legend(fontsize=7, bbox_to_anchor=(1.01, 1), loc="upper left")
    plt.tight_layout()
    savefig(fig, "trajectory", "nb04_metabolite_trajectory.png")
    traj_df.to_csv(TABLE_DIR / "metabolite_trajectory.csv")
    print(f"Trajectory computed for {len(traj_df)} metabolites across 6 stages.")
else:
    print("No significant metabolites found — run NB02 first.")


Saved figure: C:\Users\andna\Documents\KI\Results\figures\trajectory\nb04_metabolite_trajectory.png
Trajectory computed for 4 metabolites across 6 stages.


---
## 9 · MP (Metachronous Polyp) Recovery Trajectory

In [10]:
mp_traj_df = pd.DataFrame()

if sig_metabs and not stage6.empty and "MP" in stage6.values:
    traj_stats = {}
    for m in sig_metabs[:15]:
        if m not in mt_stage.columns:
            continue
        row = {}
        for s in CRC_STAGE_ORDER:
            vals = mt_stage.loc[s6_aligned == s, m].dropna()
            row[s] = {
                "median": vals.median()        if len(vals) > 0 else np.nan,
                "q25":    vals.quantile(0.25)  if len(vals) > 3 else np.nan,
                "q75":    vals.quantile(0.75)  if len(vals) > 3 else np.nan,
                "n":      len(vals),
            }
        traj_stats[m] = row                        # KEGG ID as key

    top_ids = list(traj_stats.keys())[:6]
    if top_ids:
        fig, axes = plt.subplots(2, 3, figsize=(14, 7), sharey=False)
        for ax, kid in zip(axes.flat, top_ids):
            stats   = traj_stats[kid]
            medians = [stats[s]["median"] for s in CRC_STAGE_ORDER]
            q25     = [stats[s]["q25"]    for s in CRC_STAGE_ORDER]
            q75     = [stats[s]["q75"]    for s in CRC_STAGE_ORDER]
            x_pos   = list(range(len(CRC_STAGE_ORDER)))

            ax.plot(x_pos, medians, "o-", lw=2, color="#1565C0", label="Median")
            ax.fill_between(x_pos, q25, q75, alpha=0.2, color="#1565C0", label="IQR")
            # Highlight Stage_III_IV -> MP arc in orange
            ax.plot([4, 5], [medians[4], medians[5]], "o-", lw=2.5,
                    color="#FF6F00", label="MP arc")
            ax.set_xticks(x_pos)
            ax.set_xticklabels(CRC_STAGE_ORDER, rotation=35, ha="right", fontsize=7)
            ax.set_title(nm_y.get(kid, kid)[:30], fontsize=8)
            ax.set_ylabel("Median (log10)")
            ax.legend(fontsize=6)

        # Hide unused subplots
        for ax in axes.flat[len(top_ids):]:
            ax.set_visible(False)

        plt.suptitle("MP Recovery Trajectory — Top Dysregulated Metabolites",
                     fontweight="bold")
        plt.tight_layout()
        savefig(fig, "trajectory", "nb04_mp_trajectory.png")

        mp_rows = []
        for kegg_id, row in traj_stats.items():
            for stage, vals in row.items():
                mp_rows.append({"kegg_id": kegg_id,
                                "metabolite": nm_y.get(kegg_id, kegg_id),
                                "stage": stage, **vals})
        mp_traj_df = pd.DataFrame(mp_rows)
        mp_traj_df.to_csv(TABLE_DIR / "mp_trajectory.csv", index=False)
        print(f"MP trajectory computed for {len(traj_stats)} metabolites")
    else:
        print("No trajectory stats available — check sig_metabs and stage6 data.")
else:
    print("No MP stage samples or no significant metabolites — MP trajectory skipped.")


Saved figure: C:\Users\andna\Documents\KI\Results\figures\trajectory\nb04_mp_trajectory.png
MP trajectory computed for 4 metabolites


---
## 10 · Save MOFA Results

In [11]:
mofa_results = {
    "method":             method_used,         # "MOFA+" or "TruncatedSVD fallback"
    "mofa_Z":             mofa_Z,
    "mofa_W_species":     mofa_W_species,
    "mofa_W_metabolites": mofa_W_metabolites,
    "factor_assoc":       factor_assoc,
    "sig_factors":        sig_factors,
    "var_exp_df":         var_exp_df,
    "loadings_df":        loadings_df,
    "fda_balanced_acc":   fda_bal_acc,
    "perm_stat":          perm_stat,
    "perm_pval":          perm_pval,
    "traj_df":            traj_df,
    "mp_traj_df":         mp_traj_df,
}
save_pickle(mofa_results, INTER_DIR / "mofa_results.pkl")

print("\n-- NB04 complete ---------------------------------------------------")
print(f"  Factorisation method          : {method_used}")
print(f"  Stage-associated MOFA factors : {sig_factors}")
print(f"  FDA balanced accuracy         : {fda_bal_acc:.3f}")
if perm_pval is not None:
    print(f"  PERMANOVA pseudo-F / p        : {perm_stat:.3f} / {perm_pval:.4f}")
print(f"  Trajectory metabolites        : {len(traj_df)}")
print("  -> Run NB05 (05_mediation_network.ipynb) next.")


Saved: C:\Users\andna\Documents\KI\Results\intermediate\mofa_results.pkl

-- NB04 complete ---------------------------------------------------
  Factorisation method          : MOFA+
  Stage-associated MOFA factors : ['Factor7', 'Factor4', 'Factor14']
  FDA balanced accuracy         : 0.361
  PERMANOVA pseudo-F / p        : 1.235 / 0.0580
  Trajectory metabolites        : 4
  -> Run NB05 (05_mediation_network.ipynb) next.
